# Histogram-Based Image Enhancement

This notebook uses `selfie.jpg` as the input image and applies:

- Histogram equalization
- Adaptive histogram equalization (AHE)
- Contrast limited adaptive histogram equalization (CLAHE)

> If notebook execution fails on Windows/OneDrive with a Jupyter permission error, launch Jupyter or VS Code with `JUPYTER_ALLOW_INSECURE_WRITES=1`.

For each method, the notebook plots the image and its histogram, then computes SSIM, PSNR, and MSE against the original grayscale image.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.filters import rank
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
from skimage.morphology import disk

plt.style.use('seaborn-v0_8-whitegrid')

: 

In [ ]:
IMAGE_PATH = Path('selfie.jpg')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Input image not found: {IMAGE_PATH.resolve()}')

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError('Failed to load selfie.jpg. Check that the file is a valid image.')

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

print(f'Image shape: {image_rgb.shape}')
print(f'Gray image range: {image_gray.min()} to {image_gray.max()}')

In [ ]:
def compute_metrics(reference, target):
    return {
        'SSIM': structural_similarity(reference, target, data_range=255),
        'PSNR': peak_signal_noise_ratio(reference, target, data_range=255),
        'MSE': mean_squared_error(reference, target),
    }


def plot_image_and_histograms(images_dict):
    fig, axes = plt.subplots(len(images_dict), 2, figsize=(14, 4 * len(images_dict)))
    if len(images_dict) == 1:
        axes = np.array([axes])

    for row, (title, img) in enumerate(images_dict.items()):
        axes[row, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
        axes[row, 0].set_title(title)
        axes[row, 0].axis('off')

        axes[row, 1].hist(img.ravel(), bins=256, range=(0, 256), color='black')
        axes[row, 1].set_title(f'{title} Histogram')
        axes[row, 1].set_xlabel('Pixel Intensity')
        axes[row, 1].set_ylabel('Frequency')

    plt.tight_layout()
    plt.show()


def save_result_figures(images_dict, results_dir):
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for ax, (title, img) in zip(axes.ravel(), images_dict.items()):
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    fig.savefig(results_dir / '1_enhancement_comparison.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for ax, (title, img) in zip(axes.ravel(), images_dict.items()):
        ax.hist(img.ravel(), bins=256, range=(0, 256), color='black')
        ax.set_title(f'{title} Histogram')
        ax.set_xlabel('Intensity')
        ax.set_ylabel('Frequency')
    plt.tight_layout()
    fig.savefig(results_dir / '1_histogram_comparison.png', dpi=180, bbox_inches='tight')
    plt.close(fig)


def print_metrics(metrics_dict):
    header = f"{'Method':<28} {'SSIM':>12} {'PSNR':>12} {'MSE':>12}"
    print(header)
    print('-' * len(header))
    for method, metric_values in metrics_dict.items():
        print(
            f"{method:<28} "
            f"{metric_values['SSIM']:>12.4f} "
            f"{metric_values['PSNR']:>12.4f} "
            f"{metric_values['MSE']:>12.4f}"
        )

In [ ]:
# 1. Global histogram equalization
hist_equalized = cv2.equalizeHist(image_gray)

# 2. Adaptive histogram equalization (local histogram equalization without contrast limiting)
adaptive_equalized = rank.equalize(image_gray, footprint=disk(30))

# 3. CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_equalized = clahe.apply(image_gray)

results = {
    'Original Grayscale': image_gray,
    'Histogram Equalization': hist_equalized,
    'Adaptive Histogram Equalization': adaptive_equalized,
    'CLAHE': clahe_equalized,
}

plot_image_and_histograms(results)
save_result_figures(results, RESULTS_DIR)

In [ ]:
metrics = {
    'Histogram Equalization': compute_metrics(image_gray, hist_equalized),
    'Adaptive Histogram Equalization': compute_metrics(image_gray, adaptive_equalized),
    'CLAHE': compute_metrics(image_gray, clahe_equalized),
}

print_metrics(metrics)